# Run the GTA tracking pipeline on Colab

Two-stage pipeline: **DeepEIoU tracking → GtaLink refinement**. Reads videos from Google Drive and
writes outputs back to Drive under `output/gta-track/<video>/`.

Use a **GPU runtime**: *Runtime → Change runtime type → Hardware accelerator → GPU*.

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Mount Drive and set paths

Set `VIDEO` to a clip that exists in your `input-videos/` folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/Colab Notebooks/football-analysis-project"
INPUT_DIR  = f"{DRIVE_BASE}/input_videos"
OUTPUT_DIR = f"{DRIVE_BASE}/output/gta-track"
CKPT_DIR   = f"{DRIVE_BASE}/checkpoints"
REPO       = "/content/football-analysis-2"

os.makedirs(OUTPUT_DIR, exist_ok=True)

VIDEO = f"{INPUT_DIR}/video2780-2960.mp4"     # <-- set to your clip
STEM  = Path(VIDEO).stem

# Export so the `!` shell cells below can read these:
os.environ.update(INPUT_DIR=INPUT_DIR, OUTPUT_DIR=OUTPUT_DIR,
                  CKPT_DIR=CKPT_DIR, REPO=REPO, VIDEO=VIDEO, STEM=STEM)

!ls -la "$INPUT_DIR"

## 2. Clone the code

In [ ]:
!git clone -b yolo-training https://github.com/luna4tech/football-analysis-2.git "$REPO"
%cd $REPO

## 3. Install dependencies

One shared environment serves both stages. `torch`/`torchvision` are preinstalled on Colab. Install
`requirements.txt`, then `cython_bbox` (tracker IoU) + `tqdm` + `ultralytics`, then the **vendored** `reid`
(torchreid) package Stage 1 imports — install it from the repo, **not** `pip install torchreid`
(the PyPI layout is incompatible with the vendored code).

In [ ]:
%cd $REPO
!pip install -q -r gta-link/requirements.txt
!pip install -q cython_bbox tqdm ultralytics
!pip install -q -e Deep-EIoU/Deep-EIoU/reid

## 4. Download & link the model checkpoints

Fetches `sports_model.pth.tar-60` (ReID) into your Drive `checkpoints/` folder once,
then links it and your custom `yolov11l.pt` detector checkpoint where Stage 1 expects them.

In [ ]:
import glob, os

os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_FOLDER_URL = "https://drive.google.com/drive/folders/1wItcb0yeGaxOS08_G9yRWBTnpVf0vZ2w"

have = {os.path.basename(p) for p in glob.glob(f"{CKPT_DIR}/**/*", recursive=True)}
if "sports_model.pth.tar-60" not in have:
    !pip install -q gdown
    !gdown --folder "{CKPT_FOLDER_URL}" -O "$CKPT_DIR"

dst_dir = f"{REPO}/Deep-EIoU/Deep-EIoU/checkpoints"
os.makedirs(dst_dir, exist_ok=True)
for name in ("yolov11l.pt", "sports_model.pth.tar-60"):
    matches = glob.glob(f"{CKPT_DIR}/**/{name}", recursive=True)
    assert matches, f"{name} not found under {CKPT_DIR}"
    dst = f"{dst_dir}/{name}"
    if os.path.islink(dst) or os.path.exists(dst):
        os.remove(dst)
    os.symlink(matches[0], dst)
    print("linked", matches[0], "->", dst)

!ls -la "{dst_dir}"

## 5. Run the pipeline

Runs Stage 1 (tracking) then Stage 2 (refine); outputs go to `OUTPUT_DIR/<STEM>/`.

In [ ]:
%cd $REPO
!python -m pipeline run \
    --video "$VIDEO" \
    --artifacts-dir "$OUTPUT_DIR" \
    --device gpu \
    --fp16 --fuse

Re-running is cached; to apply changed parameters add `--force-all` (or `--force-stage2`).

## 6. (Optional) Render an annotated video

The pipeline writes MOT `.txt`, not a video. Rebuild an overlay from the refined result (frames are
0-based, so no `--one_indexed`).

In [ ]:
%cd $REPO/Deep-EIoU/Deep-EIoU
!SAVE_PATH="$OUTPUT_DIR/$STEM/${STEM}_refined.mp4" && \
python tools/render_from_txt.py \
    --path "$VIDEO" \
    --txt  "$OUTPUT_DIR/$STEM/03_team/refined.txt" \
    --save_path "$SAVE_PATH" 
%cd $REPO

## 7. Verify

Track counts before vs after refinement (refined should be fewer / cleaner), then the profiling
summary.

In [ ]:
%cd $REPO/Deep-EIoU/Deep-EIoU
!python tools/count_tracks.py "$OUTPUT_DIR/$STEM/01_track/tracks.txt"
!python tools/count_tracks.py "$OUTPUT_DIR/$STEM/02_refine/refined.txt"
%cd $REPO
!cat "$OUTPUT_DIR/$STEM/profiles/summary.md"

## 8. Evaluate (HOTA / MOTA / IDF1)

The eval is **standalone and CPU-only** (it reuses this GPU session but needs no GPU). It scores the
pipeline's `refined.txt` against ground truth from Drive via **TrackEval**, reporting
**HOTA / DetA / AssA / MOTA / IDF1**.

The GT must be a **MOTChallenge `gt.txt`** (1-based frames); the pipeline's 0-based output is
converted automatically, and GT with `class=-1` is handled (no flag needed). Tip: to judge GtaLink's
refinement, watch **IDF1 + AssA** (association quality), not MOTA.

In [ ]:
!git clone -q https://github.com/JonathonLuiten/TrackEval.git /content/TrackEval
!pip install -q scipy

In [ ]:
GT_DIR = f"{DRIVE_BASE}/ground_truth"
GT = f"{GT_DIR}/gt_mot_{STEM}.txt"   # <-- EDIT to your GT file for THIS clip (MOTChallenge gt.txt)
os.environ["GT"] = GT
if not os.path.exists(GT):
    print("GT not found:", GT, "\nAvailable in", GT_DIR, ":")
    !ls -la "$GT_DIR" 2>/dev/null || echo "  (folder missing - create it and upload your gt.txt)"
    raise FileNotFoundError(GT)
print("Using GT:", GT)

In [ ]:
!grep -rl 'np\.float\|np\.int\|np\.bool' /content/TrackEval/trackeval | xargs -r sed -i 's/np\.float\b/float/g; s/np\.int\b/int/g; s/np\.bool\b/bool/g'

In [ ]:
%cd $REPO
# (--class-name defaults to 'pedestrian'; only override if your GT class label differs)
!python -m eval.evaluate \
    --pred "$OUTPUT_DIR/$STEM/02_refine/refined.txt" \
    --gt   "$GT" \
    --seq-name "$STEM" \
    --trackeval-path /content/TrackEval \
    --out  "$OUTPUT_DIR/$STEM/eval/metrics.json"
import json
print(json.dumps(json.load(open(f"{OUTPUT_DIR}/{STEM}/eval/metrics.json")), indent=2))

## Output layout

```
output/gta-track/<STEM>/
  01_track/tracks.txt       # raw tracking (MOT, 0-based frames)
  01_track/tracklets.pkl    # per-id tracklets with reused ReID features
  02_refine/refined.txt     # final refined result
  profiles/summary.md       # per-stage time / GPU / CPU / counts
  eval/metrics.json         # HOTA / DetA / AssA / MOTA / IDF1 (after step 8)
  <STEM>_refined.mp4        # only if you ran step 6
```